# FDPIS — Notebook 8: Weather at the turnaround airport, inside Layer 4

**Run notebooks 1, 3 and 7 first** (7 for the cached METAR files in `data/weather/raw/`).
Nothing here modifies `results/layer4/`, any earlier notebook, or `results/layer5/`.
Candidates go to `results/candidates/weather_l4/`.

### The claim being tested — stated so it can fail
Layer 3 weather failed (notebook 7): weather 3 h before departure says little about *new*
delay. This tests a different claim: **an aircraft that is already late absorbs less of that
lateness during a turnaround at an airport with bad weather right now** (de-icing, ground
stops, lightning ramp closures eat the slack). Per hop:

```
A_k   = D_(k-1) x r            arrival delay carried in (r = 0.895)
S_eff = Slack_k - W_k          effective slack after weather eats some of it
P_k   = max(0, A_k - S_eff)    propagated delay
D_k   = P_k + F_k              F_k = expected fresh primary delay (notebook 5)
```

`W_k` is the only new quantity. This is a weather-adjusted version of the analytical slack
model in the literature (Kafle & Zou 2016; Wang & Vaze's hybrid logistic + quantile
approach). It is **not** novel. What is ours is the timing rule and the evaluation.

### Timing — two variants, both reported (as notebook 7)
| Variant | Weather at turnaround airport X | Status |
|---|---|---|
| **T-honest** | latest METAR at X at or before the **inbound leg's actual departure** (when the inbound delay is known and the prediction is made) | what we would report |
| **T-upper** | METAR at X nearest the **inbound leg's actual arrival** (the turnaround itself) | upper bound, not deployable |

### Step 0 is a gate
If no bad-weather class shows a T-honest `W` whose 95% CI excludes 0 **and** whose point
estimate is ≥ 5 min, the notebook stops: no models are built to find an effect the raw data
does not show.

In [1]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import airportsdata

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)
WX_DIR, RAW_DIR = Path('data/weather'), Path('data/weather/raw')
CAND = Path('results/candidates/weather_l4'); CAND.mkdir(parents=True, exist_ok=True)
WX_START, WX_END = pd.Timestamp('2025-01-01', tz='UTC'), pd.Timestamp('2025-08-01', tz='UTC')
RNG = np.random.default_rng(42)
R_RECOVERY = 0.895

df = pd.read_parquet('data/processed.parquet')
AP = airportsdata.load('IATA')
stations = sorted(set(df['ORIGIN']) | set(df['DEST']))
TZ = {s: AP[s]['tz'] for s in stations}
missing = [s for s in stations if not (RAW_DIR / f'{s}.csv').exists()]
assert not missing, f'METAR cache missing for {missing} - run notebook 7 first (no re-download here)'
print(f'{len(df):,} flights | {len(stations)} stations, all cached')

1,718,426 flights | 345 stations, all cached


## 1 — METAR → per-observation features (notebook 7's exact parsing)

In [2]:
frames = []
for s in stations:
    d = pd.read_csv(RAW_DIR / f'{s}.csv', na_values=['M'], keep_default_na=False,
                    dtype={'p01i': str, 'wxcodes': str, 'skyc1': str, 'skyc2': str, 'skyc3': str})
    d['station'] = s
    frames.append(d)
obs = pd.concat(frames, ignore_index=True)
obs['valid_utc'] = pd.to_datetime(obs['valid'], utc=True).astype('datetime64[ns, UTC]')
obs = obs[(obs['valid_utc'] >= WX_START) & (obs['valid_utc'] < WX_END)]
obs = obs.sort_values(['station', 'valid_utc']).drop_duplicates(['station', 'valid_utc'], keep='last')

WX = ['VISIBILITY','WIND_KT','GUST_KT','PRECIP_IN','CEILING_FT','TEMP_F',
      'IS_THUNDER','IS_SNOW','IS_FOG','IS_IFR']
o = obs
f = pd.DataFrame({'station': o['station'], 'obs_utc': o['valid_utc']})
f['VISIBILITY'] = o['vsby'].clip(upper=10)
f['WIND_KT']    = o['sknt']
f['GUST_KT']    = o['gust'].fillna(0)
f['PRECIP_IN']  = pd.to_numeric(o['p01i'].replace('T', '0.0001'), errors='coerce')
f['TEMP_F']     = o['tmpf']
ceil = pd.Series(np.inf, index=o.index); sky_any = pd.Series(False, index=o.index)
for i in (1, 2, 3):
    c, h = o[f'skyc{i}'].fillna('').str.strip(), o[f'skyl{i}']
    sky_any |= c.ne('')
    ceil = np.where(c.isin(['BKN','OVC','VV']) & h.notna(), np.minimum(ceil, h), ceil)
f['CEILING_FT'] = pd.Series(ceil, index=o.index).replace(np.inf, 10000).clip(upper=10000).where(sky_any)
wx = o['wxcodes'].fillna('')
f['IS_THUNDER'] = wx.str.contains('TS').astype(float)
f['IS_SNOW']    = wx.str.contains('SN').astype(float)
f['IS_FOG']     = wx.str.contains('FG').astype(float)
f['IS_IFR'] = ((f['VISIBILITY'] < 3) | (f['CEILING_FT'] < 1000)).astype(float).where(
    f['VISIBILITY'].notna() | f['CEILING_FT'].notna())

# Weather class, defined once and reused everywhere. Priority TS > SN > IFR > PRECIP > CLEAR.
def wx_class(t):
    return np.select([t['IS_THUNDER'].eq(1), t['IS_SNOW'].eq(1), t['IS_IFR'].eq(1),
                      t['PRECIP_IN'].fillna(0).gt(0)],
                     ['TS', 'SN', 'IFR', 'PRECIP'], default='CLEAR')
f['WX_CLASS'] = wx_class(f)
f.loc[f[['VISIBILITY','CEILING_FT']].isna().all(axis=1) & f['IS_THUNDER'].ne(1) & f['IS_SNOW'].ne(1),
      'WX_CLASS'] = np.nan          # no visibility/ceiling and no reported TS/SN: unknown, not CLEAR
print(f'{len(f):,} observations | class mix:', f['WX_CLASS'].value_counts(dropna=False).to_dict())

2,213,152 observations | class mix: {'CLEAR': 1650053, 'IFR': 213516, 'PRECIP': 183125, 'SN': 110653, 'TS': 53265, nan: 2540}


## 2 — Hourly as-of station table (`data/weather/hourly_station.parquet`)

For each station and UTC hour H: the **latest observation at or before H:00** (within 59 min),
forward-filled ≤ 2 h — notebook 7's Table B. A lookup for time *t* uses H = floor(t), so the
observation is always at or before *t*. This is the table the API would serve from.

In [3]:
WXC = WX + ['WX_CLASS']
out = []
for s, g in f.groupby('station', sort=False):
    grid = pd.DataFrame({'hour_utc': pd.date_range(WX_START, WX_END, freq='h', inclusive='left')
                         .astype('datetime64[ns, UTC]')})
    m = pd.merge_asof(grid, g.drop(columns='station').sort_values('obs_utc'),
                      left_on='hour_utc', right_on='obs_utc', direction='backward',
                      tolerance=pd.Timedelta(minutes=59))
    m[WXC + ['obs_utc']] = m[WXC + ['obs_utc']].ffill(limit=2)
    m['station'] = s
    out.append(m)
hourly = pd.concat(out, ignore_index=True)
hourly.to_parquet(WX_DIR / 'hourly_station.parquet', index=False)
assert (hourly['obs_utc'].dropna() <= hourly.loc[hourly['obs_utc'].notna(), 'hour_utc']).all()
print(f'hourly_station.parquet: {len(hourly):,} station-hours, '
      f'{100*hourly["obs_utc"].isna().mean():.2f}% without an observation within 3 h')

hourly_station.parquet: 1,755,360 station-hours, 0.82% without an observation within 3 h


## 3 — Layer 4 candidates (notebook 3's exact filter) joined to the turnaround

Candidate = outbound leg with `CONTINUOUS_ROTATION` and `PREV_ARR_DELAY > 0` (notebook 3).
Turnaround airport X = the outbound leg's ORIGIN (= inbound DEST). Inbound times:

* inbound actual departure (UTC) = scheduled departure (local → UTC) + `DEP_DELAY`
* inbound actual arrival = that + `CRS_ELAPSED_TIME` + (`ARR_DELAY` − `DEP_DELAY`)

In [4]:
def local_to_utc(local_naive, airports):
    out = pd.Series(pd.NaT, index=local_naive.index, dtype='datetime64[ns, UTC]')
    zones = airports.map(TZ)
    for zone, idx in zones.groupby(zones).groups.items():
        out.loc[idx] = (local_naive.loc[idx].astype('datetime64[ns]')
                        .dt.tz_localize(zone, nonexistent='shift_forward', ambiguous='NaT')
                        .dt.tz_convert('UTC'))
    return out

prop = df[df['CONTINUOUS_ROTATION'].fillna(False) & (df['PREV_ARR_DELAY'] > 0)].copy()
prop['PROPAGATED_MIN'] = prop['LATE_AIRCRAFT_DELAY'].fillna(0)            # notebook 3's target

inb = df[['TAIL_NUM','FL_DATE','LEG_NUM','ORIGIN','DEST','CRS_DEP_TIME','CRS_ELAPSED_TIME',
          'DEP_DELAY','ARR_DELAY']].copy()
inb['LEG_NUM'] = inb['LEG_NUM'] + 1                                        # becomes the outbound's leg
inb = inb.rename(columns={c: f'IN_{c}' for c in ['ORIGIN','DEST','CRS_DEP_TIME','CRS_ELAPSED_TIME',
                                                  'DEP_DELAY','ARR_DELAY']})
prop = prop.merge(inb, on=['TAIL_NUM','FL_DATE','LEG_NUM'], how='left', validate='one_to_one')
assert (prop['IN_DEST'] == prop['ORIGIN']).mean() > 0.999, 'turnaround airport mismatch'
same = np.isclose(prop['IN_ARR_DELAY'], prop['PREV_ARR_DELAY'], equal_nan=True)
print(f'PREV_ARR_DELAY equals the inbound leg ARR_DELAY on {same.mean():.4%} of candidates')
assert same.mean() > 0.99, 'PREV_ARR_DELAY does not match the inbound leg'

hh = prop['IN_CRS_DEP_TIME'].astype(int)
in_dep_sched = local_to_utc(prop['FL_DATE'] + pd.to_timedelta((hh // 100) * 60 + hh % 100, unit='m'),
                            prop['IN_ORIGIN'])
prop['IN_DEP_ACTUAL_UTC'] = in_dep_sched + pd.to_timedelta(prop['IN_DEP_DELAY'], unit='m')
prop['IN_ARR_ACTUAL_UTC'] = (prop['IN_DEP_ACTUAL_UTC'] + pd.to_timedelta(prop['IN_CRS_ELAPSED_TIME'], unit='m')
                             + pd.to_timedelta(prop['IN_ARR_DELAY'] - prop['IN_DEP_DELAY'], unit='m'))
prop['X'] = prop['ORIGIN']
print(f'Candidates: {len(prop):,} | with inbound actual departure: {prop["IN_DEP_ACTUAL_UTC"].notna().sum():,}')

def honest_join(rows, t_col, prefix):
    # Latest observation at X at or before t: hourly as-of table at H = floor(t).
    key = rows[['X', t_col]].copy()
    key['hour_utc'] = key[t_col].dt.floor('h')
    m = key.merge(hourly.rename(columns={'station': 'X'}), on=['X', 'hour_utc'], how='left')
    m.index = rows.index
    return m[WXC + ['obs_utc']].add_prefix(prefix)

def nearest_join(rows, t_col, prefix):
    # Observation at X nearest t (within 90 min): the upper bound.
    out = []
    keyed = rows[['X', t_col]].dropna().reset_index()
    for s, g in keyed.groupby('X'):
        fs = f[f['station'] == s].sort_values('obs_utc')
        m = pd.merge_asof(g.sort_values(t_col), fs.drop(columns='station'), left_on=t_col,
                          right_on='obs_utc', direction='nearest', tolerance=pd.Timedelta(minutes=90))
        out.append(m.set_index('index'))
    m = pd.concat(out).reindex(rows.index)
    return m[WXC + ['obs_utc']].add_prefix(prefix)

prop = prop.join(honest_join(prop, 'IN_DEP_ACTUAL_UTC', 'H_')).join(nearest_join(prop, 'IN_ARR_ACTUAL_UTC', 'U_'))

# Honesty guarantee, per row, as notebook 7 does.
age = (prop['IN_DEP_ACTUAL_UTC'] - prop['H_obs_utc']).dropna()
assert (age >= pd.Timedelta(0)).all(), 'T-honest uses an observation newer than the prediction time'
print(f'T-honest: every observation is at or before the inbound actual departure '
      f'(age min {age.min()}, median {age.median()})')
gap = (prop['IN_ARR_ACTUAL_UTC'] - prop['U_obs_utc']).abs().dropna()
print(f'T-upper : |obs - inbound actual arrival| median {gap.median()}, max {gap.max()}')
stale = (prop['IN_ARR_ACTUAL_UTC'] - prop['H_obs_utc']).dropna()
print(f'T-honest observation age at the actual turnaround: median {stale.median()}, p90 {stale.quantile(0.9)}')
print('Coverage: T-honest', f'{prop["H_WX_CLASS"].notna().mean():.3%}', '| T-upper', f'{prop["U_WX_CLASS"].notna().mean():.3%}')

PREV_ARR_DELAY equals the inbound leg ARR_DELAY on 100.0000% of candidates


Candidates: 435,288 | with inbound actual departure: 435,288


T-honest: every observation is at or before the inbound actual departure (age min 0 days 00:00:00, median 0 days 00:37:00)
T-upper : |obs - inbound actual arrival| median 0 days 00:14:00, max 0 days 01:30:00
T-honest observation age at the actual turnaround: median 0 days 02:53:00, p90 0 days 04:44:00
Coverage: T-honest 99.740% | T-upper 99.620%


## 4 — Sanity check: is the turnaround join aligned? (notebook 7's test, re-run)

Outbound flights with `WEATHER_DELAY > 0` must show worse weather at their origin (the
turnaround airport) at the turnaround time than flights with `WEATHER_DELAY = 0`. Shift scan
−8…+8 h on the T-upper key: a timezone bug would put the peak 4–10 h away from 0.
If either check fails, the notebook stops.

In [5]:
cc = prop['WEATHER_DELAY'].notna()
wd = prop['WEATHER_DELAY'] > 0
rows = []
for v in ('U_', 'H_'):
    for c in ('IS_IFR', 'IS_THUNDER', 'VISIBILITY'):
        a, b = prop.loc[cc & wd, v + c].mean(), prop.loc[cc & ~wd, v + c].mean()
        rows.append({'variant': 'T-upper' if v == 'U_' else 'T-honest', 'feature': c,
                     'WEATHER_DELAY>0': round(a, 4), 'WEATHER_DELAY=0': round(b, 4), 'ratio': round(a / b, 2)})
sanity = pd.DataFrame(rows)
print(f'Candidates with cause codes: {cc.sum():,} | WEATHER_DELAY>0: {(cc & wd).sum():,}')
print(sanity.to_string(index=False))

s = sanity[sanity['variant'] == 'T-upper'].set_index('feature')
passed = (s.loc['IS_IFR', 'WEATHER_DELAY>0'] > s.loc['IS_IFR', 'WEATHER_DELAY=0'] and
          s.loc['IS_THUNDER', 'WEATHER_DELAY>0'] > s.loc['IS_THUNDER', 'WEATHER_DELAY=0'] and
          s.loc['VISIBILITY', 'WEATHER_DELAY>0'] < s.loc['VISIBILITY', 'WEATHER_DELAY=0'])

sub = prop.loc[cc, ['X', 'IN_ARR_ACTUAL_UTC']].dropna()
y_wd = wd[sub.index].values
scan = []
for k in range(-8, 9):
    key = pd.DataFrame({'X': sub['X'].reset_index(drop=True),
                        'hour_utc': (sub['IN_ARR_ACTUAL_UTC'] + pd.Timedelta(hours=k)).dt.floor('h')
                        .reset_index(drop=True)})          # keep the UTC dtype (.values would drop it)
    m = key.merge(hourly.rename(columns={'station': 'X'})[['X','hour_utc','IS_THUNDER']], on=['X','hour_utc'], how='left')
    th = m['IS_THUNDER'].values
    scan.append({'shift_h': k, 'thunder_ratio': round(np.nanmean(th[y_wd]) / np.nanmean(th[~y_wd]), 2)})
scan = pd.DataFrame(scan)
print('\nShift scan (thunder rate ratio, weather-delayed vs not):')
print(scan.to_string(index=False))
peak = int(scan.loc[scan['thunder_ratio'].idxmax(), 'shift_h'])
print(f'Peak at shift {peak:+d} h')
if not passed or abs(peak) >= 4:
    raise AssertionError('SANITY CHECK FAILED - the turnaround join is misaligned. Stopping.')
print('SANITY CHECK PASSED - weather-delayed outbound flights show clearly worse weather at the '
      'turnaround airport, and the separation peaks at the correct alignment.')

Candidates with cause codes: 201,679 | WEATHER_DELAY>0: 15,882
 variant    feature  WEATHER_DELAY>0  WEATHER_DELAY=0  ratio
 T-upper     IS_IFR           0.1101           0.0492   2.24
 T-upper IS_THUNDER           0.1576           0.0160   9.83
 T-upper VISIBILITY           8.7621           9.5002   0.92
T-honest     IS_IFR           0.1077           0.0617   1.75
T-honest IS_THUNDER           0.0757           0.0229   3.31
T-honest VISIBILITY           8.8590           9.3993   0.94



Shift scan (thunder rate ratio, weather-delayed vs not):
 shift_h  thunder_ratio
      -8           2.39
      -7           2.73
      -6           2.82
      -5           2.82
      -4           2.93
      -3           3.01
      -2           3.56
      -1           4.80
       0           6.83
       1          10.18
       2           7.80
       3           4.62
       4           3.15
       5           2.67
       6           2.45
       7           1.99
       8           1.82
Peak at shift +1 h
SANITY CHECK PASSED - weather-delayed outbound flights show clearly worse weather at the turnaround airport, and the separation peaks at the correct alignment.


## 5 — Step 0: does the physical effect exist? (training dates only)

Residual `R = outbound DEP_DELAY − max(0, inbound ARR_DELAY − AVAILABLE_SLACK)`. If slack fully
explained propagation, R would be fresh delay only and independent of weather at X.

* Cells: inbound arrival delay (15–30, 30–60, 60–120, 120+) × slack (0–15, 15–45, 45–90, 90+).
* Per cell, each class's mean minus `CLEAR`'s mean, with a 1,000-sample bootstrap 95% CI.
* **Absorption vs fresh delay.** Bad weather also delays the outbound flight on its own (BTS
  codes that WEATHER / NAS, not late aircraft). The *same* residual is computed for the
  late-aircraft minutes alone — `PROPAGATED_MIN − max(0, A − S)`, notebook 3's own target, on
  every candidate (uncoded = 0) — and for `WEATHER_DELAY + NAS_DELAY`. Only the late-aircraft
  part supports the absorption hypothesis.
* **`W(class)`** = pooled excess late-aircraft minutes vs `CLEAR`: the cell differences averaged
  with weights = the class's rows in each cell. Cells with fewer than 200 class rows are shrunk
  to 0 by `n / (n + 200)` (prior strength 200 pseudo-rows at 0). CI by bootstrapping rows
  within every cell, 1,000 times.
* **Gate:** some class ≠ `CLEAR` with T-honest `W` CI excluding 0 **and** `W` ≥ 5 min.

In [6]:
dates = np.sort(prop['FL_DATE'].unique())
train_end = dates[int(len(dates) * 0.70)]
val_end = dates[int(len(dates) * 0.85)]
print(f'Split (notebook 3): train <= {pd.Timestamp(train_end).date()}, val <= {pd.Timestamp(val_end).date()}, test after')
tr = prop[prop['FL_DATE'] <= train_end].copy()

base_prop = (tr['PREV_ARR_DELAY'] - tr['AVAILABLE_SLACK']).clip(lower=0)
tr['R_total'] = tr['DEP_DELAY'] - base_prop
tr['R_late']  = tr['PROPAGATED_MIN'] - base_prop
tr['R_wxnas'] = tr['WEATHER_DELAY'].fillna(0) + tr['NAS_DELAY'].fillna(0)
tr['A_BIN'] = pd.cut(tr['PREV_ARR_DELAY'], [15, 30, 60, 120, np.inf], right=False,
                     labels=['15-30', '30-60', '60-120', '120+'])
tr['S_BIN'] = pd.cut(tr['AVAILABLE_SLACK'], [-np.inf, 15, 45, 90, np.inf], right=False,
                     labels=['0-15', '15-45', '45-90', '90+'])
tr = tr[tr['A_BIN'].notna() & tr['S_BIN'].notna()]
CLASSES = ['TS', 'SN', 'IFR', 'PRECIP']
PRIOR = 200
B = 1000

def boot_means(x, B=B, chunk=100):
    # 1,000 bootstrap resample means of x (chunked to bound memory).
    x = np.asarray(x, float); n = len(x)
    out = []
    for i in range(0, B, chunk):
        k = min(chunk, B - i)
        out.append(x[RNG.integers(0, n, size=(k, n))].mean(axis=1))
    return np.concatenate(out)

def step0(d, cls_col, measure):
    # Per-cell class-minus-CLEAR differences with CIs, plus pooled shrunk W per class.
    cell_rows, W = [], []
    acc = {k: [0.0, 0.0, np.zeros(B)] for k in CLASSES}       # num, den, bootstrap num
    for (a, s), g in d.groupby(['A_BIN', 'S_BIN'], observed=True):
        ref = g.loc[g[cls_col] == 'CLEAR', measure].dropna().values
        if len(ref) < 30:
            continue
        ref_boot = boot_means(ref)                              # once per cell, shared by classes
        for k in CLASSES:
            cur = g.loc[g[cls_col] == k, measure].dropna().values
            if len(cur) == 0:
                continue
            diff = cur.mean() - ref.mean()
            bdiff = boot_means(cur) - ref_boot
            lo, hi = np.percentile(bdiff, [2.5, 97.5])
            shrink = len(cur) / (len(cur) + PRIOR) if len(cur) < PRIOR else 1.0
            cell_rows.append({'class': k, 'inbound_delay': a, 'slack': s, 'n_class': len(cur),
                              'n_clear': len(ref), 'diff_vs_clear': diff, 'ci_lo': lo, 'ci_hi': hi,
                              'shrink': shrink})
            acc[k][0] += len(cur) * shrink * diff
            acc[k][2] += len(cur) * shrink * bdiff
            acc[k][1] += len(cur)
    for k in CLASSES:
        num, den, boot_num = acc[k]
        if den:
            w, bw = num / den, boot_num / den
            W.append({'class': k, 'W_min': w, 'ci_lo': np.percentile(bw, 2.5),
                      'ci_hi': np.percentile(bw, 97.5), 'rows': int(den)})
    return pd.DataFrame(cell_rows), pd.DataFrame(W)

results, wtabs = [], []
for vname, col in (('T-honest', 'H_WX_CLASS'), ('T-upper', 'U_WX_CLASS')):
    d = tr[tr[col].notna()]
    for measure, label in (('R_total', 'total residual (DEP_DELAY)'),
                           ('R_late', 'late-aircraft residual (absorption)'),
                           ('R_wxnas', 'WEATHER + NAS minutes (fresh)')):
        t0 = time.time()
        cells_, W_ = step0(d, col, measure)
        for t_, kind in ((cells_, 'cell'), (W_, 'pooled')):
            t_.insert(0, 'measure', measure); t_.insert(0, 'variant', vname); t_.insert(0, 'level', kind)
        results.append(cells_); wtabs.append(W_)
        print(f'{vname:9} {label:38} pooled W vs CLEAR ({time.time()-t0:.0f}s):')
        print('   ' + W_[['class','W_min','ci_lo','ci_hi','rows']].round(2).to_string(index=False).replace('\n', '\n   '))

step0_df = pd.concat(results + wtabs, ignore_index=True)
step0_df.to_csv(CAND / 'step0_effect.csv', index=False)
w_table = pd.concat(wtabs, ignore_index=True)
w_table = w_table[w_table['measure'] == 'R_late'][['variant','class','W_min','ci_lo','ci_hi','rows']]
w_table.to_csv(CAND / 'w_table.csv', index=False)
print('\nW table (late-aircraft excess vs CLEAR, shrunk), saved to w_table.csv:')
print(w_table.round(2).to_string(index=False))

Split (notebook 3): train <= 2025-07-04, val <= 2025-07-18, test after


T-honest  total residual (DEP_DELAY)             pooled W vs CLEAR (1s):
    class  W_min  ci_lo  ci_hi  rows
       TS  11.16   8.90  13.29  2233
       SN   7.65   6.11   9.27  6106
      IFR   2.78   1.75   3.80  8606
   PRECIP   3.84   2.68   5.08 11769


T-honest  late-aircraft residual (absorption)    pooled W vs CLEAR (1s):
    class  W_min  ci_lo  ci_hi  rows
       TS   6.44   5.17   7.81  2233
       SN   1.30   0.22   2.38  6106
      IFR   1.36   0.75   1.92  8606
   PRECIP   1.14   0.48   1.74 11769


T-honest  WEATHER + NAS minutes (fresh)          pooled W vs CLEAR (1s):
    class  W_min  ci_lo  ci_hi  rows
       TS  10.11   8.27  12.14  2233
       SN  14.65  13.49  15.99  6106
      IFR   3.19   2.45   3.89  8606
   PRECIP   4.19   3.39   5.02 11769


T-upper   total residual (DEP_DELAY)             pooled W vs CLEAR (1s):
    class  W_min  ci_lo  ci_hi  rows
       TS  12.90  11.12  14.53  1612
       SN   9.96   8.14  12.02  5500
      IFR   5.05   3.90   6.36  7447
   PRECIP   4.92   3.84   5.98 11802


T-upper   late-aircraft residual (absorption)    pooled W vs CLEAR (1s):
    class  W_min  ci_lo  ci_hi  rows
       TS   4.23   3.52   4.94  1612
       SN   2.34   1.17   3.47  5500
      IFR   1.57   0.92   2.27  7447
   PRECIP   1.76   1.14   2.36 11802


T-upper   WEATHER + NAS minutes (fresh)          pooled W vs CLEAR (1s):
    class  W_min  ci_lo  ci_hi  rows
       TS  15.53  13.74  17.36  1612
       SN  19.96  18.39  21.61  5500
      IFR   5.76   4.71   6.79  7447
   PRECIP   5.76   5.00   6.61 11802

W table (late-aircraft excess vs CLEAR, shrunk), saved to w_table.csv:
 variant  class  W_min  ci_lo  ci_hi  rows
T-honest     TS   6.44   5.17   7.81  2233
T-honest     SN   1.30   0.22   2.38  6106
T-honest    IFR   1.36   0.75   1.92  8606
T-honest PRECIP   1.14   0.48   1.74 11769
 T-upper     TS   4.23   3.52   4.94  1612
 T-upper     SN   2.34   1.17   3.47  5500
 T-upper    IFR   1.57   0.92   2.27  7447
 T-upper PRECIP   1.76   1.14   2.36 11802


## 6 — Gate decision

In [7]:
h = w_table[w_table['variant'] == 'T-honest']
qual = h[(h['ci_lo'] > 0) & (h['W_min'] >= 5)]
GATE_PASSED = len(qual) > 0
cfg = {
    'status': 'gate_passed' if GATE_PASSED else 'gate_failed_no_effect',
    'gate_rule': 'some class != CLEAR with T-honest W 95% CI excluding 0 AND W >= 5 min',
    'timing': {'T-honest': 'latest METAR at turnaround airport at or before the inbound leg actual '
                           'departure (hourly as-of table, H = floor(t))',
               'T-upper': 'METAR at turnaround airport nearest the inbound leg actual arrival (within 90 min)'},
    'w_definition': 'pooled excess late-aircraft minutes (PROPAGATED_MIN - max(0, A - S)) vs CLEAR, '
                    'training dates only; cells with <200 class rows shrunk by n/(n+200)',
    'w_table': w_table.round(3).to_dict(orient='records'),
    'qualifying_classes': qual['class'].tolist(),
}
(CAND / 'config.json').write_text(json.dumps(cfg, indent=2, default=str))
if GATE_PASSED:
    print('GATE PASSED for', qual['class'].tolist(), '- continue to Steps 1-3.')
else:
    print('GATE FAILED - no measurable absorption effect at prediction time. '
          'Stopping: no models are built.')

GATE PASSED for ['TS'] - continue to Steps 1-3.


## 7 — Steps 1–3 setup: production Layer 4 must be reproduced first

Same split as notebook 3. Production models are reloaded from `results/layer4/` (read only)
and must reproduce **all-candidates MAE 7.67 / cascade MAE 8.35 within ±0.05**, or the
notebook stops. Features are built exactly as notebook 3 / `api/features.py`; categorical
levels come from production `config.json`.

In [8]:
assert GATE_PASSED, 'Step 0 gate failed - Steps 1-3 must not run'
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, roc_auc_score

L4 = Path('results/layer4')
cfg4 = json.loads((L4 / 'config.json').read_text())
PROP_FEATURES, CAT_COLS, PROD_THR = cfg4['features'], cfg4['cat_cols'], float(cfg4['gate_threshold'])
CAT_TYPES = {c: pd.CategoricalDtype(categories=cfg4['category_levels'][c]) for c in CAT_COLS}
prod_gate = xgb.XGBClassifier(); prod_gate.load_model(str(L4 / 'xgb_gate.json'))
prod_q = {}
for n in ('lower', 'median', 'upper'):
    m = xgb.XGBRegressor(); m.load_model(str(L4 / f'xgb_hurdle_{n}.json')); prod_q[n] = m
arr_counts = df.groupby(['DEST', 'FL_DATE', 'ARR_HOUR']).size()

CLASSES_ALL = ['TS', 'SN', 'IFR', 'PRECIP', 'CLEAR']
WXF = ['VISIBILITY', 'CEILING_FT', 'GUST_KT', 'PRECIP_IN', 'IS_THUNDER', 'IS_SNOW']
ONEHOT = [f'WX_{c}' for c in CLASSES_ALL]
W_LOOKUP = {v: dict(zip(g['class'], g['W_min'])) | {'CLEAR': 0.0}
            for v, g in w_table.groupby('variant')}
print('W(class) used by R and H (train-only, shrunk):', {k: {c: round(x, 2) for c, x in v.items()}
                                                        for k, v in W_LOOKUP.items()})

def l4_matrix(rows, inbound, slack):
    # Notebook 3 / api.features.build_layer4_matrix, with AVAILABLE_SLACK replaceable.
    f_ = rows.copy()
    f_['AVAILABLE_SLACK'] = np.asarray(slack, float)
    f_['PREV_ARR_DELAY'] = np.asarray(inbound, float)
    f_['SLACK_DEFICIT'] = f_['PREV_ARR_DELAY'] - f_['AVAILABLE_SLACK']
    f_['DELAY_TO_SLACK_RATIO'] = (f_['PREV_ARR_DELAY'] / f_['AVAILABLE_SLACK'].clip(lower=1)).clip(upper=50)
    est = (f_['PREV_ARR_DELAY'] - f_['AVAILABLE_SLACK']).clip(lower=0)
    f_['PROJECTED_ARR_HOUR'] = (((f_['CRS_ARR_MIN'] + est) // 60) % 24).astype(int)
    f_['PROJECTED_HOUR_ARRIVALS'] = (pd.MultiIndex.from_arrays(
        [f_['DEST'], f_['FL_DATE'], f_['PROJECTED_ARR_HOUR']]).map(arr_counts).astype(float).fillna(0))
    f_['ARRIVAL_CONGESTION_DELTA'] = f_['PROJECTED_HOUR_ARRIVALS'] - f_['DEST_HOUR_ARRIVALS']
    X = f_[PROP_FEATURES].copy()
    for c in CAT_COLS:
        X[c] = X[c].astype(CAT_TYPES[c])
    return X

def wx_block(cls, vals):
    # Turnaround weather features: class one-hot + raw METAR fields.
    b = pd.DataFrame({f'WX_{c}': (np.asarray(cls) == c).astype(float) for c in CLASSES_ALL})
    for c in WXF:
        b[c] = np.asarray(vals[c], float)
    return b

ptrain = prop[prop['FL_DATE'] <= train_end].copy()
pval   = prop[(prop['FL_DATE'] > train_end) & (prop['FL_DATE'] <= val_end)].copy()
ptest  = prop[prop['FL_DATE'] > val_end].copy()
TARGET = 'PROPAGATED_MIN'

def hurdle_eval(p, q, y, thr):
    # Notebook 3's final: median where gate >= thr; symmetric interval at thr.
    lo, mid, hi = q
    final = np.where(p >= thr, mid, 0.0)
    flo, fhi = np.where(p >= thr, lo, 0.0), np.where(p >= thr, hi, 0.0)
    return final, flo, fhi

def sorted_q(models, X):
    st = np.vstack([np.clip(models[n].predict(X), 0, None) for n in ('lower', 'median', 'upper')])
    st.sort(axis=0)
    return st[0], st[1], st[2]

def metrics(name, y, final, lo, hi, p=None):
    nz = y > 0
    return {'candidate': name, 'mae_all': mean_absolute_error(y, final),
            'mae_cascades': mean_absolute_error(y[nz], final[nz]),
            'gate_auc': roc_auc_score(nz.astype(int), p) if p is not None else np.nan,
            'coverage': float(((y >= lo) & (y <= hi)).mean()), 'mean_width': float((hi - lo).mean())}

yte = ptest[TARGET].values
Xte_prod = l4_matrix(ptest, ptest['PREV_ARR_DELAY'], ptest['AVAILABLE_SLACK'])
p_prod = prod_gate.predict_proba(Xte_prod)[:, 1]
q_prod = sorted_q(prod_q, Xte_prod)
base = metrics('production', yte, *hurdle_eval(p_prod, q_prod, yte, PROD_THR), p_prod)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in base.items()})
for k, exp in (('mae_all', 7.67), ('mae_cascades', 8.35)):
    if abs(base[k] - exp) > 0.05:
        raise AssertionError(f'Production Layer 4 not reproduced: {k} {base[k]:.3f} vs {exp}. Stopping.')
print(f'Production reproduced: all {base["mae_all"]:.2f}, cascades {base["mae_cascades"]:.2f} '
      f'(coverage {base["coverage"]:.1%}) at gate threshold {PROD_THR}.')

W(class) used by R and H (train-only, shrunk): {'T-honest': {'TS': 6.44, 'SN': 1.3, 'IFR': 1.36, 'PRECIP': 1.14, 'CLEAR': 0.0}, 'T-upper': {'TS': 4.23, 'SN': 2.34, 'IFR': 1.57, 'PRECIP': 1.76, 'CLEAR': 0.0}}


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


{'candidate': 'production', 'mae_all': 7.6663, 'mae_cascades': 8.3531, 'gate_auc': 0.9272, 'coverage': 0.702, 'mean_width': 11.8067}
Production reproduced: all 7.67, cascades 8.35 (coverage 70.2%) at gate threshold 0.3.


## 8 — Step 1: R, the rule

`S_eff = AVAILABLE_SLACK − W(class)` with the train-only W table.
* **R** — feed `S_eff` through the **existing** production hurdle in place of `AVAILABLE_SLACK`
  (derived slack features recomputed from it). No retraining; production threshold.
* **R-analytic** — the pure rule `max(0, A − S_eff)`, no model at all (reported for reference;
  its no-weather twin is notebook 3's arithmetic baseline).

In [9]:
VARIANTS = {'T-honest': 'H_', 'T-upper': 'U_'}
rows_cmp, preds = [base], {'production': {'final': hurdle_eval(p_prod, q_prod, yte, PROD_THR)[0], 'p': p_prod}}
arith = (ptest['PREV_ARR_DELAY'] - ptest['AVAILABLE_SLACK']).clip(lower=0).values
rows_cmp.append(metrics('arithmetic (no weather)', yte, arith, arith, arith))

def w_of(rows, v):
    return rows[VARIANTS[v] + 'WX_CLASS'].map(W_LOOKUP[v]).fillna(0.0).values

for v in VARIANTS:
    s_eff = ptest['AVAILABLE_SLACK'].values - w_of(ptest, v)
    X = l4_matrix(ptest, ptest['PREV_ARR_DELAY'], s_eff)
    p = prod_gate.predict_proba(X)[:, 1]
    q = sorted_q(prod_q, X)
    fin, lo, hi = hurdle_eval(p, q, yte, PROD_THR)
    rows_cmp.append(metrics(f'R {v}', yte, fin, lo, hi, p)); preds[f'R {v}'] = {'final': fin, 'p': p}
    ra = np.clip(ptest['PREV_ARR_DELAY'].values - s_eff, 0, None)
    rows_cmp.append(metrics(f'R-analytic {v}', yte, ra, ra, ra)); preds[f'R-analytic {v}'] = {'final': ra}
print(pd.DataFrame(rows_cmp).round(3).to_string(index=False))

C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


              candidate  mae_all  mae_cascades  gate_auc  coverage  mean_width
             production    7.666         8.353     0.927     0.702      11.807
arithmetic (no weather)    8.473        11.716       NaN     0.467       0.000
             R T-honest    7.706         8.360     0.927     0.701      11.853
    R-analytic T-honest    8.477        11.635       NaN     0.463       0.000
              R T-upper    7.690         8.338     0.928     0.702      11.854
     R-analytic T-upper    8.455        11.601       NaN     0.463       0.000


## 9 — Step 2: M, retrained with turnaround weather

Gate + three quantile models, notebook 3's hyperparameters, `PROP_FEATURES` + turnaround weather
(class one-hot, visibility, ceiling, gust, precip, thunder, snow). The gate threshold is
re-derived on **validation** with notebook 3's rule (minimise validation cascade MAE over
0.3…0.7). A no-weather retrain is run too, to measure retraining noise against production.

In [10]:
QUANT = {'lower': 0.10, 'median': 0.50, 'upper': 0.90}

def fit_hurdle(Xtr, ytr, Xv, yv):
    wtr, wv = (ytr > 0).astype(int), (yv > 0).astype(int)
    spw = (wtr == 0).sum() / (wtr == 1).sum()
    gate = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=10, scale_pos_weight=spw,
                             enable_categorical=True, tree_method='hist', eval_metric='auc',
                             early_stopping_rounds=40, random_state=42, n_jobs=-1)
    gate.fit(Xtr, wtr, eval_set=[(Xv, wv)], verbose=False)
    q = {}
    for n, a in QUANT.items():
        m = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=a, n_estimators=600,
                             max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             min_child_weight=10, enable_categorical=True, tree_method='hist',
                             early_stopping_rounds=50, random_state=42, n_jobs=-1)
        m.fit(Xtr[(ytr > 0)], ytr[ytr > 0], eval_set=[(Xv[(yv > 0)], yv[yv > 0])], verbose=False)
        q[n] = m
    # notebook 3's threshold rule, on validation
    pv, qv = gate.predict_proba(Xv)[:, 1], sorted_q(q, Xv)
    nzv, best = yv > 0, None
    for t in (0.3, 0.4, 0.5, 0.6, 0.7):
        mae = mean_absolute_error(yv[nzv], np.where(pv >= t, qv[1], 0.0)[nzv])
        if best is None or mae < best[1]:
            best = (t, mae)
    return gate, q, best[0]

def mats(d, v=None):
    X = l4_matrix(d, d['PREV_ARR_DELAY'], d['AVAILABLE_SLACK'])
    if v is None:
        return X
    blk = wx_block(d[VARIANTS[v] + 'WX_CLASS'].values, {c: d[VARIANTS[v] + c].values for c in WXF})
    blk.index = X.index
    return pd.concat([X, blk], axis=1)

ytr, yv = ptrain[TARGET].values, pval[TARGET].values
models = {}
for name, v in (('retrain (no weather)', None), ('M T-honest', 'T-honest'), ('M T-upper', 'T-upper')):
    t0 = time.time()
    g, q, thr = fit_hurdle(mats(ptrain, v), ytr, mats(pval, v), yv)
    Xt = mats(ptest, v)
    p = g.predict_proba(Xt)[:, 1]
    fin, lo, hi = hurdle_eval(p, sorted_q(q, Xt), yte, thr)
    rows_cmp.append(metrics(name, yte, fin, lo, hi, p)); preds[name] = {'final': fin, 'p': p}
    models[name] = (g, q, thr, v)
    print(f'{name:22} threshold (validation) {thr} | {time.time()-t0:.0f}s')
print(pd.DataFrame(rows_cmp).round(3).to_string(index=False))

C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


retrain (no weather)   threshold (validation) 0.3 | 60s


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


M T-honest             threshold (validation) 0.3 | 85s


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


M T-upper              threshold (validation) 0.3 | 88s
              candidate  mae_all  mae_cascades  gate_auc  coverage  mean_width
             production    7.666         8.353     0.927     0.702      11.807
arithmetic (no weather)    8.473        11.716       NaN     0.467       0.000
             R T-honest    7.706         8.360     0.927     0.701      11.853
    R-analytic T-honest    8.477        11.635       NaN     0.463       0.000
              R T-upper    7.690         8.338     0.928     0.702      11.854
     R-analytic T-upper    8.455        11.601       NaN     0.463       0.000
   retrain (no weather)    7.666         8.353     0.927     0.702      11.807
             M T-honest    7.638         8.373     0.927     0.703      11.810
              M T-upper    7.616         8.353     0.928     0.704      11.690


## 10 — Step 3: H, hybrid (rule + learned residual)

Target = actual propagated delay − R-analytic. Features = M's + the rule prediction + `W(class)`.
Monotone constraints: + on inbound arrival delay, − on `AVAILABLE_SLACK`, + on `W(class)`.
Three quantile models of the residual; final = rule + residual, clipped at 0. No gate (so no
gate AUC).

In [11]:
MONO = {'PREV_ARR_DELAY': 1, 'AVAILABLE_SLACK': -1, 'W_CLASS': 1}

def h_mats(d, v):
    X = mats(d, v)
    w = w_of(d, v)
    X['W_CLASS'] = w
    X['RULE_PRED'] = np.clip(d['PREV_ARR_DELAY'].values - (d['AVAILABLE_SLACK'].values - w), 0, None)
    return X

for v in VARIANTS:
    t0 = time.time()
    Xtr_, Xv_, Xt_ = h_mats(ptrain, v), h_mats(pval, v), h_mats(ptest, v)
    rtr, rv = ytr - Xtr_['RULE_PRED'].values, yv - Xv_['RULE_PRED'].values
    hq = {}
    for n, a in QUANT.items():
        m = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=a, n_estimators=600,
                             max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             min_child_weight=10, enable_categorical=True, tree_method='hist',
                             monotone_constraints=MONO, early_stopping_rounds=50, random_state=42, n_jobs=-1)
        m.fit(Xtr_, rtr, eval_set=[(Xv_, rv)], verbose=False)
        hq[n] = m
    st = np.vstack([np.clip(Xt_['RULE_PRED'].values + hq[n].predict(Xt_), 0, None)
                    for n in ('lower', 'median', 'upper')]); st.sort(axis=0)
    rows_cmp.append(metrics(f'H {v}', yte, st[1], st[0], st[2]))
    preds[f'H {v}'] = {'final': st[1]}
    models[f'H {v}'] = (None, hq, None, v)
    print(f'H {v}: {time.time()-t0:.0f}s')

comparison = pd.DataFrame(rows_cmp)
for c in ('mae_all', 'mae_cascades'):
    comparison[f'delta_{c}'] = comparison[c] - base[c]
comparison.to_csv(CAND / 'comparison.csv', index=False)
print(comparison.round(3).to_string(index=False))

C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


H T-honest: 161s


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


H T-upper: 165s
              candidate  mae_all  mae_cascades  gate_auc  coverage  mean_width  delta_mae_all  delta_mae_cascades
             production    7.666         8.353     0.927     0.702      11.807          0.000               0.000
arithmetic (no weather)    8.473        11.716       NaN     0.467       0.000          0.807               3.363
             R T-honest    7.706         8.360     0.927     0.701      11.853          0.040               0.007
    R-analytic T-honest    8.477        11.635       NaN     0.463       0.000          0.811               3.282
              R T-upper    7.690         8.338     0.928     0.702      11.854          0.024              -0.015
     R-analytic T-upper    8.455        11.601       NaN     0.463       0.000          0.788               3.248
   retrain (no weather)    7.666         8.353     0.927     0.702      11.807          0.000               0.000
             M T-honest    7.638         8.373     0.927     0.703      

## 11 — Segmented MAE: actual turnaround weather class

Rows are segmented by the **actual** weather at the turnaround (T-upper class) — the conditions
the aircraft really met — for every candidate, so all are judged on the same rows. `CLEAR`
dominates the test set; the point of this layer is the bad-weather rows.

In [12]:
seg_rows = []
seg_lab = ptest['U_WX_CLASS'].fillna('UNKNOWN').values
nz = yte > 0
for name, pr in preds.items():
    for cls in CLASSES_ALL:
        m = seg_lab == cls
        if not m.any():                     # e.g. no snow at all in the late-July test dates
            seg_rows.append({'candidate': name, 'class': cls, 'rows': 0, 'mae_all': np.nan, 'mae_cascades': np.nan})
            continue
        seg_rows.append({'candidate': name, 'class': cls, 'rows': int(m.sum()),
                         'mae_all': mean_absolute_error(yte[m], pr['final'][m]),
                         'mae_cascades': mean_absolute_error(yte[m & nz], pr['final'][m & nz]) if (m & nz).any() else np.nan})
segments = pd.DataFrame(seg_rows)
segments = segments.merge(segments[segments['candidate'] == 'production'][['class', 'mae_all', 'mae_cascades']]
                          .rename(columns={'mae_all': 'prod_mae_all', 'mae_cascades': 'prod_mae_cascades'}), on='class')
segments['delta_mae_all'] = segments['mae_all'] - segments['prod_mae_all']
segments['delta_mae_cascades'] = segments['mae_cascades'] - segments['prod_mae_cascades']
segments.to_csv(CAND / 'segments.csv', index=False)
print(segments.pivot(index='candidate', columns='class', values='delta_mae_cascades').round(2).to_string())
print('\n(delta cascade MAE vs production; negative = better). Rows per class:',
      segments[segments['candidate'] == 'production'].set_index('class')['rows'].to_dict())

class                 CLEAR   IFR  PRECIP  SN    TS
candidate                                          
H T-honest             2.53  3.18    2.74 NaN  3.51
H T-upper              2.51  2.77    2.27 NaN  1.10
M T-honest             0.03 -0.29    0.05 NaN  0.11
M T-upper              0.07 -0.30   -0.18 NaN -0.78
R T-honest             0.00 -0.06    0.05 NaN  0.06
R T-upper              0.00 -0.19   -0.04 NaN -0.31
R-analytic T-honest    3.21  2.66    3.76 NaN  4.63
R-analytic T-upper     3.27  2.46    3.34 NaN  2.91
production             0.00  0.00    0.00 NaN  0.00
retrain (no weather)   0.00  0.00    0.00 NaN  0.00

(delta cascade MAE vs production; negative = better). Rows per class: {'TS': 1776, 'SN': 0, 'IFR': 870, 'PRECIP': 4431, 'CLEAR': 65514}


## 12 — Multi-hop: notebook 5's integration with each candidate plugged in

Seeds, frontier, expected primary delay, SHRINK and recovery exactly as notebook 5 (taken from
the API bundle, which reproduces notebook 5). Weather per hop:
* **T-honest** — each downstream turnaround airport's observation **as of the seed time** (the
  seed leg's actual departure): nothing newer exists when the prediction is made, so it gets
  staler with depth.
* **T-upper** — each turnaround's own observation nearest its actual time.

Production must reproduce notebook 5's total-delay MAE by depth.

In [13]:
from api.models import load_bundle
B = load_bundle()
test = df[df['FL_DATE'] > val_end].copy().sort_values(['TAIL_NUM', 'FL_DATE', 'LEG_NUM']).reset_index(drop=True)
test['ROT_ID'] = test['TAIL_NUM'].astype(str) + '_' + test['FL_DATE'].dt.strftime('%Y%m%d')
# turnaround weather for every test leg (T-upper: nearest the inbound leg's actual arrival)
tl = test.merge(inb, on=['TAIL_NUM', 'FL_DATE', 'LEG_NUM'], how='left')
hh = tl['IN_CRS_DEP_TIME'].fillna(0).astype(int)
dep_s = local_to_utc(tl['FL_DATE'] + pd.to_timedelta((hh // 100) * 60 + hh % 100, unit='m'),
                     tl['IN_ORIGIN'].fillna(tl['ORIGIN']))
tl['IN_ARR_ACTUAL_UTC'] = (dep_s + pd.to_timedelta(tl['IN_DEP_DELAY'], unit='m')
                           + pd.to_timedelta(tl['IN_CRS_ELAPSED_TIME'], unit='m')
                           + pd.to_timedelta(tl['IN_ARR_DELAY'] - tl['IN_DEP_DELAY'], unit='m'))
tl['X'] = tl['ORIGIN']
up = nearest_join(tl, 'IN_ARR_ACTUAL_UTC', 'U_')
up.index = test.index
test = pd.concat([test, up], axis=1)
hh = test['CRS_DEP_TIME'].astype(int)
test['DEP_ACTUAL_UTC'] = (local_to_utc(test['FL_DATE'] + pd.to_timedelta((hh // 100) * 60 + hh % 100, unit='m'),
                                       test['ORIGIN']) + pd.to_timedelta(test['DEP_DELAY'], unit='m'))
test_idx = test.set_index(['ROT_ID', 'LEG_NUM']).sort_index()
test['IS_ORIGIN'] = ((test['ARR_DELAY'] >= 15) & (test['LATE_AIRCRAFT_DELAY'].fillna(0) == 0)
                     & (test['LEG_NUM'] < test['TAIL_LEGS_TODAY']))
origins = test[test['IS_ORIGIN']]
hourly_idx = hourly.set_index(['station', 'hour_utc']).sort_index()

def honest_at(rows, seed_time):
    t = pd.DatetimeIndex(seed_time)
    t = t.tz_localize('UTC') if t.tz is None else t.tz_convert('UTC')   # .values drops tz; restore it
    key = pd.MultiIndex.from_arrays([rows['ORIGIN'].values, t.floor('h').astype('datetime64[ns, UTC]')])
    got = hourly_idx.reindex(key)
    return got['WX_CLASS'].values, {c: got[c].values for c in WXF}

def make_predictor(name):
    if name == 'production':
        def f_(rows, inbound, cls, vals):
            X = l4_matrix(rows, inbound, rows['AVAILABLE_SLACK'])
            return prod_gate.predict_proba(X)[:, 1], sorted_q(prod_q, X), PROD_THR
        return f_
    kind, v = name.split(' ', 1)
    if kind == 'R':
        def f_(rows, inbound, cls, vals):
            w = pd.Series(cls).map(W_LOOKUP[v]).fillna(0.0).values
            X = l4_matrix(rows, inbound, rows['AVAILABLE_SLACK'].values - w)
            return prod_gate.predict_proba(X)[:, 1], sorted_q(prod_q, X), PROD_THR
        return f_
    g, q, thr, _ = models[name]
    if kind == 'M':
        def f_(rows, inbound, cls, vals):
            X = l4_matrix(rows, inbound, rows['AVAILABLE_SLACK'])
            blk = wx_block(cls, vals); blk.index = X.index
            X = pd.concat([X, blk], axis=1)
            return g.predict_proba(X)[:, 1], sorted_q(q, X), thr
        return f_
    def f_(rows, inbound, cls, vals):                      # H
        X = l4_matrix(rows, inbound, rows['AVAILABLE_SLACK'])
        blk = wx_block(cls, vals); blk.index = X.index
        X = pd.concat([X, blk], axis=1)
        w = pd.Series(cls).map(W_LOOKUP[v]).fillna(0.0).values
        X['W_CLASS'] = w
        X['RULE_PRED'] = np.clip(np.asarray(inbound) - (rows['AVAILABLE_SLACK'].values - w), 0, None)
        st = np.vstack([np.clip(X['RULE_PRED'].values + q[n].predict(X), 0, None)
                        for n in ('lower', 'median', 'upper')]); st.sort(axis=0)
        return np.ones(len(X)), (st[0], st[1], st[2]), 0.0
    return f_

def traverse(name):
    pred = make_predictor(name)
    timing = None if name == 'production' else name.split(' ', 1)[1]
    fr = pd.DataFrame({'ROT_ID': origins['ROT_ID'].values, 'cur_leg': origins['LEG_NUM'].values,
                       'inbound_delay': origins['ARR_DELAY'].values,
                       'seed_time': origins['DEP_ACTUAL_UTC'].values})
    out = []
    for depth in range(1, 6):
        fr = fr.assign(next_leg=fr['cur_leg'] + 1)
        look = test_idx[test_idx.index.isin(list(zip(fr['ROT_ID'], fr['next_leg'])))].reset_index()
        mg = fr.merge(look, left_on=['ROT_ID', 'next_leg'], right_on=['ROT_ID', 'LEG_NUM'], how='inner')
        mg = mg[mg['CONTINUOUS_ROTATION'].fillna(False)].reset_index(drop=True)
        if mg.empty:
            break
        if timing == 'T-honest':
            cls, vals = honest_at(mg, mg['seed_time'])
        else:
            cls, vals = mg['U_WX_CLASS'].values, {c: mg['U_' + c].values for c in WXF}
        p, (lo, mid, hi), thr = pred(mg, mg['inbound_delay'].values, cls, vals)
        mid = np.where(p >= thr, mid, 0.0)
        prim = B.expected_primary(mg)
        mg['depth'] = depth
        mg['pred_prop'], mg['pred_total'] = mid, mid + prim * B.shrink
        mg['actual_prop'] = mg['LATE_AIRCRAFT_DELAY'].fillna(0)
        mg['actual_total'] = mg['DEP_DELAY_NEW'].fillna(0)
        mg['turn_class'] = mg['U_WX_CLASS']
        out.append(mg[['depth', 'pred_prop', 'pred_total', 'actual_prop', 'actual_total', 'turn_class']])
        chain = mid + prim
        alive = chain > 0
        fr = pd.DataFrame({'ROT_ID': mg.loc[alive, 'ROT_ID'].values, 'cur_leg': mg.loc[alive, 'LEG_NUM'].values,
                           'inbound_delay': chain[alive] * B.recovery_rate,
                           'seed_time': mg.loc[alive, 'seed_time'].values})
    return pd.concat(out, ignore_index=True)

mh_rows = []
for name in ['production'] + [n for n in preds if n.split(' ')[0] in ('R', 'M', 'H')]:
    t0 = time.time()
    ch = traverse(name)
    for dpt, g in ch.groupby('depth'):
        cas = g['actual_prop'] > 0
        mh_rows.append({'candidate': name, 'depth': dpt, 'n': len(g),
                        'mae_total': mean_absolute_error(g['actual_total'], g['pred_total']),
                        'corr_total': np.corrcoef(g['pred_total'], g['actual_total'])[0, 1],
                        'mae_inherited_cascades': mean_absolute_error(g.loc[cas, 'actual_prop'], g.loc[cas, 'pred_prop'])})
    print(f'{name:14} traversed ({time.time()-t0:.0f}s)')
multihop = pd.DataFrame(mh_rows)
prod_mh = multihop[multihop['candidate'] == 'production'].set_index('depth')
print('\nProduction depth table (must match notebook 5: total MAE 20.87 / 31.88 / 35.15 / 38.66 / 39.91):')
print(prod_mh[['n', 'mae_total', 'corr_total', 'mae_inherited_cascades']].round(3).to_string())
expected = [20.87, 31.88, 35.15, 38.66, 39.91]
if not np.allclose(prod_mh['mae_total'].values[:5], expected, atol=0.05):
    raise AssertionError('Notebook 5 integration not reproduced. Stopping.')
multihop = multihop.merge(prod_mh[['mae_total', 'mae_inherited_cascades']].add_prefix('prod_').reset_index(), on='depth')
multihop['delta_mae_total'] = multihop['mae_total'] - multihop['prod_mae_total']
multihop.to_csv(CAND / 'multihop.csv', index=False)
print(multihop.pivot(index='candidate', columns='depth', values='delta_mae_total').round(2).to_string())
print('\nDepth-3 total-delay correlation by candidate (boundary rule: below 0.33 -> not quantified):')
print(multihop[multihop['depth'] == 3].set_index('candidate')['corr_total'].round(3).to_string())

C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


production     traversed (2s)


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


...

H T-honest     traversed (2s)


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])
C:\Users\Shrish Singh Sourya\AppData\Local\Temp\ipykernel_14020\2882814999.py:37: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = X[c].astype(CAT_TYPES[c])


H T-upper      traversed (2s)

Production depth table (must match notebook 5: total MAE 20.87 / 31.88 / 35.15 / 38.66 / 39.91):
           n  mae_total  corr_total  mae_inherited_cascades
depth                                                      
1      21999     20.867       0.674                   8.668
2      13167     31.884       0.428                  26.247
3       7234     35.148       0.327                  33.893
4       3421     38.659       0.233                  36.433
5       1377     39.908       0.226                  42.532
depth          1     2     3     4     5
candidate                               
H T-honest  1.20 -0.13 -1.43 -2.52 -3.21
H T-upper   0.95 -0.36 -1.56 -2.64 -3.29
M T-honest  0.00  0.01 -0.03 -0.10 -0.15
M T-upper  -0.04 -0.09 -0.10 -0.05 -0.21
R T-honest -0.01  0.03  0.03  0.02  0.05
R T-upper  -0.01 -0.02 -0.00 -0.02  0.03
production  0.00  0.00  0.00  0.00  0.00

Depth-3 total-delay correlation by candidate (boundary rule: below 0.33 -> not qua

## 13 — Feature importance, and the promotion rule

**Promotion rule (fixed before results):** a candidate is promoted only if it is **T-honest**,
improves cascade MAE by **≥ 0.3 min** over production, is **not worse than production by more
than 0.1 min on `CLEAR`** (cascade MAE, actual-class segment), and **does not worsen depth-1 or
depth-2 integrated MAE**. If several qualify, prefer the simplest (R over H over M).

In [14]:
imps = []
for name in ('M T-honest', 'M T-upper'):
    g, q, thr, v = models[name]
    imps.append(pd.DataFrame({'candidate': name, 'model': 'magnitude (median)',
                              'feature': q['median'].get_booster().feature_names,
                              'importance': q['median'].feature_importances_}))
    imps.append(pd.DataFrame({'candidate': name, 'model': 'gate', 'feature': g.get_booster().feature_names,
                              'importance': g.feature_importances_}))
for name in ('H T-honest', 'H T-upper'):
    _, hq, _, _ = models[name]
    imps.append(pd.DataFrame({'candidate': name, 'model': 'residual (median)',
                              'feature': hq['median'].get_booster().feature_names,
                              'importance': hq['median'].feature_importances_}))
imp = pd.concat(imps, ignore_index=True)
imp['is_weather'] = imp['feature'].isin(ONEHOT + WXF + ['W_CLASS', 'RULE_PRED'])
imp.to_csv(CAND / 'feature_importance.csv', index=False)
for (c, mdl), g in imp.groupby(['candidate', 'model']):
    g = g.sort_values('importance', ascending=False).reset_index(drop=True)
    wx_ = g[g['is_weather']]
    print(f'{c} / {mdl}: weather share {wx_["importance"].sum():.3f}; top weather features:',
          [(r.feature, int(r.Index) + 1) for r in wx_.head(4).itertuples()])

decisions = []
for name in comparison['candidate']:
    kind = name.split(' ')[0]
    if kind not in ('R', 'M', 'H'):
        continue
    row = comparison.set_index('candidate').loc[name]
    clear = segments[(segments['candidate'] == name) & (segments['class'] == 'CLEAR')].iloc[0]
    mh = multihop[(multihop['candidate'] == name) & (multihop['depth'].isin([1, 2]))]
    checks = {'t_honest': name.endswith('T-honest'),
              'cascade_gain_ge_0.3': row['delta_mae_cascades'] <= -0.3,
              'clear_not_worse_0.1': clear['delta_mae_cascades'] <= 0.1,
              'depth1_2_not_worse': bool((mh['delta_mae_total'] <= 0).all()) if len(mh) == 2 else False}
    decisions.append({'candidate': name, **checks, 'qualifies': all(checks.values()),
                      'delta_mae_cascades': row['delta_mae_cascades'],
                      'clear_delta_cascades': clear['delta_mae_cascades'],
                      'depth1_delta': mh.set_index('depth')['delta_mae_total'].get(1, np.nan),
                      'depth2_delta': mh.set_index('depth')['delta_mae_total'].get(2, np.nan)})
dec = pd.DataFrame(decisions)
print(dec.round(3).to_string(index=False))
order = {'R': 0, 'H': 1, 'M': 2}
qual = dec[dec['qualifies']].assign(o=lambda d: d['candidate'].str.split(' ').str[0].map(order)).sort_values('o')
PROMOTED = qual['candidate'].iloc[0] if len(qual) else None
cfg = json.loads((CAND / 'config.json').read_text())
cfg.update({'status': f'promoted: {PROMOTED}' if PROMOTED else 'no candidate promoted',
            'promotion_rule': 'T-honest; cascade MAE improves >= 0.3 min vs production; CLEAR (actual class) '
                              'cascade MAE not worse by > 0.1 min; depth-1 and depth-2 integrated MAE not worse. '
                              'Prefer R over H over M.',
            'decisions': dec.round(4).to_dict(orient='records'),
            'production_reference': {'mae_all': base['mae_all'], 'mae_cascades': base['mae_cascades']}})
(CAND / 'config.json').write_text(json.dumps(cfg, indent=2, default=str))
for name, (g, q, thr, v) in models.items():
    tag = name.replace(' ', '_').replace('(', '').replace(')', '')
    if g is not None:
        g.save_model(str(CAND / f'{tag}_gate.json'))
    for n, m in q.items():
        m.save_model(str(CAND / f'{tag}_{n}.json'))
print('\nPROMOTED:', PROMOTED if PROMOTED else 'none - nothing is promoted, which is a valid result.')

H T-honest / residual (median): weather share 0.079; top weather features: [('RULE_PRED', 3), ('WX_CLEAR', 12), ('IS_SNOW', 13), ('WX_SN', 14)]
H T-upper / residual (median): weather share 0.130; top weather features: [('RULE_PRED', 3), ('W_CLASS', 6), ('WX_SN', 8), ('IS_SNOW', 10)]
M T-honest / gate: weather share 0.094; top weather features: [('IS_SNOW', 10), ('WX_SN', 11), ('IS_THUNDER', 14), ('PRECIP_IN', 15)]
M T-honest / magnitude (median): weather share 0.114; top weather features: [('WX_SN', 12), ('IS_THUNDER', 13), ('IS_SNOW', 14), ('WX_TS', 15)]
M T-upper / gate: weather share 0.129; top weather features: [('WX_CLEAR', 9), ('WX_SN', 10), ('IS_SNOW', 11), ('IS_THUNDER', 13)]
M T-upper / magnitude (median): weather share 0.137; top weather features: [('IS_SNOW', 6), ('WX_CLEAR', 10), ('WX_SN', 11), ('IS_THUNDER', 14)]
 candidate  t_honest  cascade_gain_ge_0.3  clear_not_worse_0.1  depth1_2_not_worse  qualifies  delta_mae_cascades  clear_delta_cascades  depth1_delta  depth2_delt


PROMOTED: none - nothing is promoted, which is a valid result.
